In [ ]:
!pip install scipy

In [ ]:
# Clear, minimal imports used in the lesson examples
from pprint import pprint

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import pyreadstat
    _HAS_PYREADSTAT = True
except Exception:  # ImportError or other import-time problems
    pyreadstat = None
    _HAS_PYREADSTAT = False

from scipy import stats


In [ ]:
# Helper to load SPSS .sav dataset using pyreadstat and return the dataframe + metadata.
def load_sav(path):
    """Load an SPSS .sav file and return (df, metadata).

    If pyreadstat is not available, raise a clear error explaining how to
    install it. This makes the notebook easier to debug in environments
    where pyreadstat isn't preinstalled.
    """
    if not _HAS_PYREADSTAT:
        raise RuntimeError("pyreadstat is required to load SPSS .sav files. Install it with `pip install pyreadstat`.")
    df, metadata = pyreadstat.read_sav(path)
    return df, metadata

# Path is relative to this notebook location in the lesson
data_path = '../../data/0_raw/ZM_LFS_DATASET2024_Annual_10percent.sav'
print('Loading data from:', data_path)
df, metadata = load_sav(data_path)

# %% [Select columns of interest]
# This cell selected human-readable question labels from the SAV file metadata
# and maps them back to the underlying column names in the dataframe.
cols = [
    # --- Demographics & Background ---
    "Is ... Male or Female?",
    "How old was ... at (his/her) last birthday?",
    "What is the highest grade/level of education that ... has successfully completed?",
    "What is ...'s current marital status?",
    "What is ...'s relationship to the head of the household?",
    "1. Province",
    "2. District",

    # --- Employment & Work ---
    "In the main job/business that (NAME) has, is she/he...",
    "INDUSTRY",
    "Occupation",
    "How many hours does (NAME) usually work per week in his/her...? Main job",
    "How many hours does (NAME) usually work per week in his/her...? OVERALL TOTAL",
    "What is the frequency of .....'s income/earnings in his/her main job?",
    "Would (NAME) want to work more hours per week than usually worked, provided the extra hours are paid?",
    "Is ?. employed on the basis of a written contract or an oral agreement?",

    # --- Income & Earnings ---
    "What is your annually/monthly/weekly/daily/hourly wage or salary before deductions?",
    "What are your annual/monthly/weekly/daily/hourly earnings after expenses?",
    "At what age did NAME start work for the first time in his /her life",

    # --- Time Use: Household Activities ---
    "During the last 7 days how much time did  (NAME) spend on Cleaning the house, washing clothes, cooking or shopping for the household",
    "During the last 7 days how much time did  (NAME) spend on Fetching water from natural or public sources for use by the household",
    "During the last 7 days how much time did (NAME) spend on Collecting firewood or other natural products for use as fuel by the household",
    "In the last 7 days, how much time did (NAME) spend on Leisure e.g., playing sports, watching TV etc.?",
    "In the last 7 days, how much time did (NAME) spend on Personal care e.g bathing, eating and sleeping?",
    "In the last 7 days how much time did name spend travelling from home to\xa0place\xa0of\xa0work",

    # --- Time Use: Hours by Day of Week (Main Job) ---
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Monday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Tuesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Wednesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Thursday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Friday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Saturday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Sunday Main job?",

    # --- Financial Inclusion ---
    "P.20. Do you own a mobile phone",
    "P.21. Do you have a mobile money account in your own name",
    "P.23. How often do you use mobile money?",
    "P.25.  On a scale of 1 to 4, Do you find mobile money services to be cheap or expensive?",
    "P.30.A Savings at a bank",
    "P.30.G.Savings with savings group",
    "P.30.D.Savings that you keep on your mobile phone",
    "What method do you mainly use to pay for food/groceries?",

    # --- Education ---
    "Can... read and write in any language?",
    "Has... ever attended school?",
    "Is (NAME) currently attending school?",
    "Have (NAME) ever repeated any level of schooling any point in time?",
    "At what age did (NAME) begin school?",
]

# Map human-readable labels to the dataset column names using metadata
names_to_labels = metadata.column_names_to_labels
names_to_labels_reduced = {}
selected_column_names = []
for label in cols:
    for colname, collabel in names_to_labels.items():
        if collabel == label:
            selected_column_names.append(colname)
            names_to_labels_reduced[colname] = collabel

print('Mapped the following labels to column names:')
pprint(names_to_labels_reduced)

# Filter the dataframe to the selected columns (keeps the dataset smaller for lesson examples)
df = df[selected_column_names]

# %% [Extract value labels from metadata]
variable_value_labels = metadata.variable_value_labels


In [ ]:
# Create a frozen normal distribution object (mean=0, sd=1) and sample from it
standard_normal = stats.norm(loc=0, scale=1)

# Draw a large random sample to show the familiar bell curve
sample = standard_normal.rvs(size=100000, random_state=42)

# Quick visual: histogram using pandas convenience method
pd.DataFrame(sample, columns=['x']).hist(bins=50)

# Show the CDF at 0 (should be ~0.5 for a standard normal)
print('standard_normal.cdf(0) =', standard_normal.cdf(0))

# --- New cell: plot theoretical PDF and CDF and compare to the sample ---
# Plot the PDF over a grid and overlay the sample histogram (density=True)
x_grid = np.linspace(-4, 4, 400)
pdf_vals = standard_normal.pdf(x_grid)
cdf_vals = standard_normal.cdf(x_grid)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: histogram (density) + theoretical PDF
axes[0].hist(sample, bins=50, density=True, alpha=0.4, color='C0', label='sample (density)')
axes[0].plot(x_grid, pdf_vals, 'r-', lw=2, label='theoretical PDF (N(0,1))')
axes[0].set_title('Sample histogram (density) and theoretical PDF')
axes[0].set_xlabel('x')
axes[0].set_ylabel('Density')
axes[0].legend()

# Right: theoretical CDF and empirical CDF from the sample
# Empirical CDF (simple step function)
sorted_sample = np.sort(sample)
ecdf_y = np.arange(1, len(sorted_sample) + 1) / len(sorted_sample)
axes[1].plot(x_grid, cdf_vals, 'r-', lw=2, label='theoretical CDF (N(0,1))')
axes[1].step(sorted_sample, ecdf_y, where='post', color='C0', alpha=0.6, label='empirical CDF (sample)')
axes[1].set_title('Theoretical CDF vs Empirical CDF')
axes[1].set_xlabel('x')
axes[1].set_ylabel('CDF')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# We'll compute a 95% t-based confidence interval for the average age (A3) by province (PROV).
# Keep the example minimal and clear: select the relevant columns and drop missing values.

df_interval = df[[ 'A2', 'A3', 'PROV' ]].dropna()
confidence_level = 0.95

# Helper to compute a t-based confidence interval for a 1-D numeric array/series
def t_confidence_interval(series, confidence=0.95):
    """Return (mean, lower_bound, upper_bound, n) for the given numeric series."""
    arr = np.asarray(series)
    n = arr.size
    mean = np.mean(arr)
    se = stats.sem(arr)
    dfree = n - 1
    # If n is very small or se is zero, stats.t.interval may behave unexpectedly; handle simply
    if n <= 1 or se == 0:
        return mean, mean, mean, n
    low, high = stats.t.interval(confidence, df=dfree, loc=mean, scale=se)
    return mean, low, high, n

for prov in df_interval['PROV'].unique():
    province_data = df_interval[df_interval['PROV'] == prov]
    mean_age, low, high, n = t_confidence_interval(province_data['A3'], confidence=confidence_level)
    print(f"Province {prov!s}: mean age={mean_age:.2f}, CI{int(confidence_level*100)}%=({low:.2f}, {high:.2f}), n={n}")


In [ ]:
# Small example to show why geometric mean matters for multiplicative processes (growth factors)
baseline = 100
year_1_growth = 1.5   # +50%
year_2_growth = 0.5   # -50% (i.e. multiply by 0.5)

# The actual multiplicative result after two years is the product of the growth factors
product_result = baseline * year_1_growth * year_2_growth
print('Baseline:', baseline)
print('After year 1 factor and year 2 factor product result:', product_result)

# Arithmetic (classical) mean of the factors — not appropriate to apply twice for multiplicative processes
arithmetic_mean = (year_1_growth + year_2_growth) / 2
print('Arithmetic mean of factors:', arithmetic_mean)
print('If you naively applied arithmetic mean twice: baseline * arithmetic_mean * arithmetic_mean =', baseline * arithmetic_mean * arithmetic_mean)

# Geometric mean of the factors — appropriate for multiplicative processes
gmean = stats.gmean([year_1_growth, year_2_growth])
print('Geometric mean of factors:', gmean)
print('Applying geometric mean twice (approx product of factors):', baseline * gmean * gmean)
print('Direct product vs geometric-mean approximation difference:', product_result - (baseline * gmean * gmean))


In [ ]:
def descriptive_report(data, name="Variable"):
    """Print a compact descriptive statistics report for a numeric array/series.

    Shows common summary statistics, a trimmed mean, and standard error.
    """
    arr = np.asarray(data)
    # Use scipy.stats.describe for consistent outputs
    desc = stats.describe(arr, nan_policy='omit')

    print(f"=== Descriptive Report: {name} ===")
    print(f"N:               {desc.nobs}")
    print(f"Mean:            {desc.mean:,.2f}")
    print(f"5% Trimmed Mean: {stats.trim_mean(arr, 0.05):,.2f}")
    print(f"Median:          {np.median(arr):,.2f}")
    print(f"Std Dev:         {np.sqrt(desc.variance):,.2f}")
    print(f"Min:             {desc.minmax[0]:,.2f}")
    print(f"Max:             {desc.minmax[1]:,.2f}")
    print(f"Skewness:        {desc.skewness:.3f}")
    print(f"Excess Kurtosis: {desc.kurtosis:.3f}")
    print(f"Std Error:       {stats.sem(arr):,.2f}")

# Example usage of the descriptive report on age (A3) for the whole dataset (after dropping missing values)
age_series = df['A3'].dropna()
descriptive_report(age_series, name='Age (A3)')

# Also demonstrate descriptive_report on the simulated standard normal sample used earlier
# (shows the same statistics functions but on a known distribution)
descriptive_report(sample, name='Simulated standard normal sample')
